# ⚡ Previsão de Carga Horária — SIN

## Contexto

As três análises anteriores responderam perguntas descritivas: como o consumo
evoluiu ao longo de décadas, como o perfil diário se comporta hora a hora,
e como a temperatura se relaciona com a demanda. Este notebook fecha o ciclo
com uma pergunta preditiva: **dado o que sabemos, conseguimos prever a carga
das próximas horas?**

A resposta não parte do zero. Cada feature deste modelo tem uma análise que
a justifica — o lag de 3 horas de temperatura vem da análise de correlação
intradiária, o lag de 7 dias captura a sazonalidade semanal identificada no
perfil horário, e os feriados aparecem como anomalias no calendar heatmap.
Modelagem orientada por análise, não por tentativa e erro.

## Dados

- **Carga:** ONS — dados horários por subsistema (2019–2026)
- **Temperatura:** Open-Meteo — séries horárias por capital representativa de cada subsistema
- **Split temporal:** treino até dez/2025, teste em 2026

## Estrutura

Três modelos em ordem crescente de complexidade:

**Seasonal Naive** — benchmark: a carga desta hora é igual à mesma hora
da semana passada. Qualquer modelo que não superar isso não serve.

**Regressão Linear** — primeiro modelo real com todas as features.
Estabelece o ganho que vem de incluir temperatura, calendário e lags.

**XGBoost** — modelo principal. Captura não-linearidades e interações
entre features que a regressão linear não consegue — como a relação
assimétrica entre temperatura e carga no Sul, ou o efeito diferente de
feriados por região.



In [ ]:
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── CONSTANTES ────────────────────────────────────────────────────────
REGIOES_COORDS = {
    'SUDESTE':  {'lat': -23.55, 'lon': -46.63, 'cidade': 'São Paulo'},
    'NORDESTE': {'lat':  -3.73, 'lon': -38.52, 'cidade': 'Fortaleza'},
    'NORTE':    {'lat':  -3.10, 'lon': -60.02, 'cidade': 'Manaus'},
    'SUL':      {'lat': -30.03, 'lon': -51.23, 'cidade': 'Porto Alegre'},
}

CORES_REG = {
    'SUDESTE':  '#636EFA',
    'NORDESTE': '#EF553B',
    'NORTE':    '#00CC96',
    'SUL':      '#AB63FA',
}


## 1. Carregamento e Merge

*Carrega dados do ONS e Open-Meteo, merge por timestamp e região.*

In [ ]:
# ── Carregamento ──────────────────────────────────────────────────────
df_carga = pd.read_parquet('data/carga_diaria_ons_2019_2026.parquet')
df_temp  = pd.read_parquet('data/temperatura_openmeteo_2019_2026.parquet')

# ── Padronização de nomes ─────────────────────────────────────────────
df_carga['regiao'] = df_carga['nom_subsistema'].str.replace(
    'SUDESTE/CENTRO-OESTE', 'SUDESTE', regex=False)

# ── Merge por timestamp + região ──────────────────────────────────────
df = (df_carga
      .merge(
          df_temp[['time', 'temperature_2m', 'regiao']],
          left_on=['din_instante', 'regiao'],
          right_on=['time', 'regiao'],
          how='inner')
      .drop(columns=['time', 'nom_subsistema', 'id_subsistema'])
      .rename(columns={
          'din_instante':           'timestamp',
          'val_cargaenergiahomwmed': 'carga_mwmed',
          'temperature_2m':          'temperatura_c',
      }))

df['hora'] = df['timestamp'].dt.hour
df['mes']  = df['timestamp'].dt.month
df['data'] = df['timestamp'].dt.date

print(f"Shape: {df.shape}")
print(f"Período: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"Nulos: {df.isna().sum().sum()}")
df.head()


## 2. Baseline — Seasonal Naive

*Previsão mais simples: mesma hora da semana passada. Benchmark mínimo a superar.*

In [ ]:
# ==============================================================================
# ── SEASONAL NAIVE — BASELINE ──────────────────────────────────────────────────
# ==============================================================================
# Previsão: carga desta hora = mesma hora exatamente 7 dias atrás
# É o benchmark mínimo que qualquer modelo precisa superar

results_naive = {}

# Guardar testes para plotagem
tests_plot = {}

for regiao in df['regiao'].unique():
    df_reg = df[df['regiao'] == regiao].sort_values('timestamp').copy()

    # Link exato de 7 dias atrás
    df_aux = df_reg[['timestamp', 'carga_mwmed']].rename(
        columns={'carga_mwmed': 'pred_naive'}
    )

    df_reg['ts_7d'] = df_reg['timestamp'] - pd.Timedelta(days=7)

    df_reg = df_reg.merge(
        df_aux,
        left_on='ts_7d',
        right_on='timestamp',
        how='left',
        suffixes=('', '_drop')
    )

    df_reg = df_reg.drop(columns=['ts_7d', 'timestamp_drop'])

    # Split temporal — treino até 2025, teste em 2026
    test = df_reg[
        df_reg['timestamp'] >= '2026-01-01'
    ].dropna(subset=['pred_naive'])

    # Métricas
    mae = mean_absolute_error(
        test['carga_mwmed'],
        test['pred_naive']
    )

    mape = np.mean(
        np.abs(
            (test['carga_mwmed'] - test['pred_naive'])
            / test['carga_mwmed']
        )
    ) * 100

    results_naive[regiao] = {
        'MAE': round(mae, 1),
        'MAPE (%)': round(mape, 2)
    }

    # Guardar para gráfico
    tests_plot[regiao] = test.copy()

df_naive = pd.DataFrame(results_naive).T
df_naive.loc['MÉDIA'] = df_naive.mean()

print("=== Seasonal Naive ===")
print(df_naive)

# ==============================================================================
# ── GRÁFICOS: REAL vs PREVISTO ─────────────────────────────────────────────────
# ==============================================================================

regioes = list(tests_plot.keys())

fig = make_subplots(
    rows=len(regioes),
    cols=1,
    subplot_titles=regioes,
    shared_xaxes=False,
    vertical_spacing=0.05
)

for i, regiao in enumerate(regioes, start=1):

    # Primeiras 2 semanas de 2026
    plot_df = tests_plot[regiao][
        tests_plot[regiao]['timestamp'] <= '2026-01-15'
    ]

    # Série real
    fig.add_trace(
        go.Scatter(
            x=plot_df['timestamp'],
            y=plot_df['carga_mwmed'],
            mode='lines',
            name=f'{regiao} - Real',
            line=dict(width=2)
        ),
        row=i,
        col=1
    )

    # Série prevista
    fig.add_trace(
        go.Scatter(
            x=plot_df['timestamp'],
            y=plot_df['pred_naive'],
            mode='lines',
            name=f'{regiao} - Naive',
            line=dict(dash='dash')
        ),
        row=i,
        col=1
    )

fig.update_layout(
    height=700,
    width=1000,
    title='Seasonal Naive — Real vs Previsto (Jan/2026)',
    template='plotly_dark',
    showlegend=False
)

fig.show()


![Gráfico de Dispersão de Temperatura](midia/naive.png)

### Observações

O Seasonal Naive captura bem o ritmo geral — picos noturnos, vales de
madrugada, diferença entre dia útil e fim de semana. Com MAPE médio de
5.7%, é um benchmark respeitável para um modelo sem nenhuma variável
explicativa.

Mas o dia 8 de janeiro expõe a principal limitação do modelo: é feriado
(Dia de Nossa Senhora Aparecida em alguns estados) e o Seasonal Naive
usou como referência o mesmo dia da semana anterior — uma quinta-feira
normal. O resultado é um erro sistemático de vários milhares de MWmed,
visível no gráfico como a maior divergência de toda a quinzena.

O Sul apresenta o maior erro (8.9% MAPE) — é a região com mais
variabilidade sazonal, onde um verão atípico ou uma frente fria derruba
a previsão baseada em semana anterior.

Essa limitação é estrutural: qualquer modelo que use apenas o passado
recente como referência vai falhar em feriados, eventos climáticos
extremos e qualquer situação que quebre o padrão semanal. Para superar
isso, o modelo precisa de variáveis explicativas — calendário, temperatura
e histórico de longo prazo.


## 3. Feature Engineering

O Seasonal Naive mostrou onde o modelo simples falha: feriados, clima
atípico e qualquer quebra do padrão semanal. As features abaixo foram
construídas diretamente a partir dos insights das análises anteriores —
não são escolhas arbitrárias.

**Calendário:** hora do dia, dia da semana, mês, fim de semana e feriado.
A variável de feriado resolve o erro sistemático do Seasonal Naive no
dia 8 de janeiro.

**Cíclicas:** hora e dia da semana representados como seno e cosseno —
para que o modelo entenda que 23h e 0h são adjacentes, não opostos.

**Lags de carga:** 24h (ontem mesma hora), 168h (semana passada mesma
hora) e média móvel 24h. Capturam a memória de curto e médio prazo
do sistema.

**Lags de temperatura:** 3h e 24h — os lags com maior correlação
identificados na análise de sensibilidade térmica.

Split temporal: treino até dez/2025, teste em jan/2026 em diante.
Nenhum dado futuro vaza para o treino.

In [ ]:
import holidays
from sklearn.preprocessing import LabelEncoder

df_model = df.copy().sort_values(['regiao', 'timestamp']).reset_index(drop=True)

# ── Variáveis temporais ───────────────────────────────────────────────
df_model['dia_semana'] = df_model['timestamp'].dt.dayofweek
df_model['mes']        = df_model['timestamp'].dt.month
df_model['é_fds']      = (df_model['dia_semana'] >= 5).astype(int)

# Cíclicas — evita que XGBoost trate 23h e 0h como distantes
df_model['hora_sin'] = np.sin(2 * np.pi * df_model['hora'] / 24)
df_model['hora_cos'] = np.cos(2 * np.pi * df_model['hora'] / 24)
df_model['dia_sin']  = np.sin(2 * np.pi * df_model['dia_semana'] / 7)
df_model['dia_cos']  = np.cos(2 * np.pi * df_model['dia_semana'] / 7)

# ── Feriados nacionais ────────────────────────────────────────────────
br_holidays = holidays.Brazil(years=range(2019, 2027))
df_model['é_feriado'] = df_model['timestamp'].dt.date.apply(
    lambda x: 1 if x in br_holidays else 0)

# ── Lags de carga (por região) ────────────────────────────────────────
for regiao in df_model['regiao'].unique():
    mask = df_model['regiao'] == regiao
    s    = df_model.loc[mask, 'carga_mwmed']
    df_model.loc[mask, 'carga_lag_24h']  = s.shift(24)
    df_model.loc[mask, 'carga_lag_168h'] = s.shift(168)
    df_model.loc[mask, 'carga_roll_24h'] = s.shift(1).rolling(24).mean()

# ── Lags de temperatura — justificados pela análise de lag ───────────
for regiao in df_model['regiao'].unique():
    mask = df_model['regiao'] == regiao
    t    = df_model.loc[mask, 'temperatura_c']
    df_model.loc[mask, 'temp_lag_3h']  = t.shift(3)
    df_model.loc[mask, 'temp_lag_24h'] = t.shift(24)

# ── Encoding ──────────────────────────────────────────────────────────
le = LabelEncoder()
df_model['regiao_enc'] = le.fit_transform(df_model['regiao'])
df_model = df_model.dropna().reset_index(drop=True)

FEATURES = [
    'hora', 'hora_sin', 'hora_cos',
    'dia_semana', 'dia_sin', 'dia_cos',
    'mes', 'é_fds', 'é_feriado',
    'temperatura_c', 'temp_lag_3h', 'temp_lag_24h',
    'carga_lag_24h', 'carga_lag_168h', 'carga_roll_24h',
    'regiao_enc'
]

TARGET = 'carga_mwmed'

# Split temporal
split = pd.Timestamp('2026-01-01')
train = df_model[df_model['timestamp'] <  split]
test  = df_model[df_model['timestamp'] >= split].copy()

print(f"Features: {len(FEATURES)}")
print(f"Treino: {len(train):,} | Teste: {len(test):,}")
print(f"Período teste: {test['timestamp'].min()} → {test['timestamp'].max()}")


Com as features construídas, o primeiro modelo real pode ser treinado.
A regressão linear serve como referência — ela captura relações lineares
entre as features e a carga, mas não consegue modelar interações
complexas como o efeito diferente de temperatura por região e horário.

## 4. Regressão Linear

*Primeiro modelo real — todas as features, relação linear. Referência antes do XGBoost.*

In [ ]:


X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])
pipe_lr.fit(X_train, y_train)
test['pred_lr'] = pipe_lr.predict(X_test)

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

results_lr = {}
for regiao in test['regiao'].unique():
    sub = test[test['regiao'] == regiao]
    results_lr[regiao] = {
        'MAE':      round(mean_absolute_error(sub[TARGET], sub['pred_lr']), 1),
        'MAPE (%)': round(mape(sub[TARGET], sub['pred_lr']), 2)
    }

df_lr = pd.DataFrame(results_lr).T
df_lr.loc['MÉDIA'] = df_lr.mean()
print("=== Regressão Linear ===")
print(df_lr)
print("\n=== Ganho vs Seasonal Naive (MAPE) ===")
print((df_naive['MAPE (%)'] - df_lr['MAPE (%)']).round(2))


# Pegar as regiões únicas presentes no teste
regioes = test['regiao'].unique()

fig = make_subplots(
    rows=len(regioes),
    cols=1,
    subplot_titles=regioes,
    shared_xaxes=True,  # Deixa o zoom do eixo X sincronizado para todas as regiões
    vertical_spacing=0.06
)

for i, regiao in enumerate(regioes, start=1):
    
    # Filtrar os dados da região até dia 15 de Janeiro para manter a mesma janela
    plot_df = test[
        (test['regiao'] == regiao) & 
        (test['timestamp'] <= '2026-01-15')
    ].sort_values('timestamp').copy()

    # 1. Linha da Carga Real (Atual)
    fig.add_trace(
        go.Scatter(
            x=plot_df['timestamp'],
            y=plot_df['carga_mwmed'],
            mode='lines',
            name='Real',
            line=dict(color='#1f77b4', width=2), # Azul clássico para o Real
            legendgroup='real',
            showlegend=(i == 1) # Mostra a legenda apenas no primeiro subgráfico
        ),
        row=i,
        col=1
    )

    # 2. Linha da Previsão da Regressão Linear
    fig.add_trace(
        go.Scatter(
            x=plot_df['timestamp'],
            y=plot_df['pred_lr'],
            mode='lines',
            name='Regressão Linear',
            line=dict(color='#ff7f0e', width=2, dash='dot'), # Laranja pontilhado para a previsão
            legendgroup='lr',
            showlegend=(i == 1)
        ),
        row=i,
        col=1
    )

# Configurações do Layout
fig.update_layout(
    height=600,
    title='Carga Real vs Previsão — Regressão Linear (Jan/2026)',
    template='plotly_dark',
    hovermode='x unified', # Mostra os valores de todas as linhas ao passar o mouse
    margin=dict(l=50, r=50, t=80, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Ajustar títulos dos eixos Y com as unidades
for i in range(1, len(regioes) + 1):
    fig.update_yaxes(title_text="MWmed", row=i, col=1)

fig.show()



![Gráfico de Dispersão de Temperatura](midia/linearreg.png)


### Observações

A regressão linear reduz o MAPE médio de 5.7% para 4.6% — ganho de 1.1pp
sobre o Seasonal Naive. As features de calendário e temperatura estão
funcionando.

O **Sudeste** tem o maior ganho absoluto (1.87pp) — é a região onde
temperatura e calendário são mais determinantes, e o modelo linear consegue
capturar essa relação.

O **Sul** tem o maior ganho de todos (3.45pp) — exatamente a região com
maior erro no Naive. As features de temperatura resolveram parte do problema.

O **Norte** é o caso problemático: o modelo linear ficou **pior** que o
Naive (-1.78pp). A relação entre as features e a carga no Norte não é
linear — o modelo está introduzindo erro ao tentar forçar uma relação
que não existe dessa forma.

Olhando os gráficos, o padrão geral é capturado — a regressão acompanha
os ciclos diários e semanais. Mas nos picos extremos e nos vales mais
profundos, a linha pontilhada sistematicamente subestima ou superestima.
A relação não é linear, e o modelo linear não tem como saber disso.

O Norte revelou o limite estrutural da regressão linear: quando a relação
entre features e carga envolve interações complexas — como o efeito
diferente de temperatura por horário e região — um modelo linear não
consegue aprender. O XGBoost não tem essa limitação: ele constrói árvores
de decisão que capturam interações e não-linearidades automaticamente,
sem precisar que o analista as especifique manualmente.

## 5. XGBoost


In [ ]:
# ==============================================================================
# ── X / y ──────────────────────────────────────────────────────────────────────
# ==============================================================================

X_train = train[FEATURES].astype(float)
y_train = train[TARGET]

X_test  = test[FEATURES].astype(float)
y_test  = test[TARGET]

# ==============================================================================
# ── MODELO XGBOOST ─────────────────────────────────────────────────────────────
# ==============================================================================

xgb_model = XGBRegressor(

    objective='reg:squarederror',

    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    n_jobs=-1
)

# Treinar
xgb_model.fit(X_train, y_train)

# Prever
test['pred_xgb'] = xgb_model.predict(X_test)

# ==============================================================================
# ── MÉTRICAS ───────────────────────────────────────────────────────────────────
# ==============================================================================

def mape(y_true, y_pred):

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.mean(
        np.abs((y_true - y_pred) / y_true)
    ) * 100

results_xgb = {}

for regiao in test['regiao'].unique():

    sub = test[test['regiao'] == regiao]

    results_xgb[regiao] = {

        'MAE': round(
            mean_absolute_error(
                sub[TARGET],
                sub['pred_xgb']
            ),
            1
        ),

        'MAPE (%)': round(
            mape(
                sub[TARGET],
                sub['pred_xgb']
            ),
            2
        )
    }

# ==============================================================================
# ── RESULTADOS ────────────────────────────────────────────────────────────────
# ==============================================================================

df_xgb = pd.DataFrame(results_xgb).T
df_xgb.loc['MÉDIA'] = df_xgb.mean()

print("=== XGBoost ===")
print(df_xgb)

# Comparação com baseline
if 'df_naive' in globals():

    print("\n=== Ganho vs Seasonal Naive (MAPE) ===")

    ganho = (
        df_naive['MAPE (%)']
        - df_xgb['MAPE (%)']
    ).round(2)

    print(ganho)

# Comparação com regressão linear
if 'df_lr' in globals():

    print("\n=== Ganho vs Regressão Linear (MAPE) ===")

    ganho_lr = (
        df_lr['MAPE (%)']
        - df_xgb['MAPE (%)']
    ).round(2)

    print(ganho_lr)

    # Pegar as regiões únicas presentes no teste
regioes = test['regiao'].unique()

fig = make_subplots(
    rows=len(regioes),
    cols=1,
    subplot_titles=regioes,
    shared_xaxes=True,  # Mantém o zoom do eixo X sincronizado
    vertical_spacing=0.06
)

for i, regiao in enumerate(regioes, start=1):
    
    # Filtrar os dados da região até dia 15 de Janeiro para manter a mesma janela
    plot_df = test[
        (test['regiao'] == regiao) & 
        (test['timestamp'] <= '2026-01-15')
    ].sort_values('timestamp').copy()

    # 1. Linha da Carga Real (Atual)
    fig.add_trace(
        go.Scatter(
            x=plot_df['timestamp'],
            y=plot_df['carga_mwmed'],
            mode='lines',
            name='Real',
            line=dict(color='#1f77b4', width=2), # Azul clássico para o Real
            legendgroup='real',
            showlegend=(i == 1)
        ),
        row=i,
        col=1
    )

    # 2. Linha da Previsão do XGBoost
    fig.add_trace(
        go.Scatter(
            x=plot_df['timestamp'],
            y=plot_df['pred_xgb'],
            mode='lines',
            name='XGBoost',
            line=dict(color='#00cc96', width=2, dash='dot'), # Verde pontilhado para o XGBoost
            legendgroup='xgb',
            showlegend=(i == 1)
        ),
        row=i,
        col=1
    )

# Configurações do Layout
fig.update_layout(
    height=600,
    title='Carga Real vs Previsão — XGBoost Regressor (Jan/2026)',
    template='plotly_dark',
    hovermode='x unified', # Mostra os valores de todas as linhas ao passar o mouse
    margin=dict(l=50, r=50, t=80, b=50),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Ajustar títulos dos eixos Y com as unidades
for i in range(1, len(regioes) + 1):
    fig.update_yaxes(title_text="MWmed", row=i, col=1)

fig.show()

![Gráfico de Dispersão de Temperatura](midia/boost.png)

### Observações

O salto de performance é imediato e visível. As linhas pontilhadas colam
na curva real de forma que a regressão linear não conseguia — os picos
são capturados, os vales são respeitados, e a forma geral da curva é
preservada mesmo nos dias mais atípicos.

O MAPE médio cai de 4.65% para 2.77% — quase metade do erro da regressão
linear. O **Sudeste** chega a 2.04%, resultado expressivo para previsão
de carga horária com dado real.

O **Norte**, que foi o caso problemático da regressão linear, é resolvido
pelo XGBoost: de 5.24% para 2.31%. A relação não-linear entre as features
e a carga nessa região — que o modelo linear não conseguia aprender — é
capturada pelas árvores de decisão sem nenhuma especificação manual.

O **Sul** tem o maior ganho absoluto (5.04pp vs Naive) — exatamente a
região mais complexa, com comportamento bimodal verão/inverno. É onde
a capacidade do XGBoost de capturar interações entre temperatura, mês
e hora faz mais diferença.

Os erros que persistem estão concentrados nos primeiros dias de janeiro —
o modelo foi treinado até dezembro e os primeiros dias de 2026 têm
comportamento ligeiramente diferente do padrão histórico. É o efeito
de deriva temporal natural em qualquer modelo de série temporal.

O XGBoost claramente supera os modelos anteriores — mas a pergunta
seguinte é: por quê? Quais features foram mais determinantes para essa
performance? A análise de importância de features fecha o ciclo do
projeto, conectando os resultados do modelo com os insights das análises
anteriores.

In [ ]:
import plotly.graph_objects as go
import pandas as pd

importancias_weight = xgb_model.get_booster().get_score(importance_type='weight')


dados_peso = []
for i, feat in enumerate(FEATURES):
    nome_f = f"f{i}"
    peso = importancias_weight.get(feat, importancias_weight.get(nome_f, 0))
    dados_peso.append({'Feature': feat, 'Importancia': peso})

df_weight = pd.DataFrame(dados_peso)
df_weight = df_weight.sort_values(by='Importancia', ascending=True)

# 2. Plotar o novo gráfico
fig_weight = go.Figure(go.Bar(
    x=df_weight['Importancia'],
    y=df_weight['Feature'],
    orientation='h',
    marker_color='#00cc96'
))

fig_weight.update_layout(
    title='Importância das Features — XGBoost (Métrica de Frequência / Weight)',
    xaxis_title='Número de vezes que a feature foi usada nas árvores',
    yaxis_title='Variáveis (Features)',
    template='plotly_dark',
    height=600,
    margin=dict(l=150, r=40, t=60, b=50)
)

fig_weight.show()

![Gráfico de Dispersão de Temperatura](midia/features.png)

### Observações

O gráfico de importância fecha o ciclo analítico do projeto — cada feature
relevante tem uma análise anterior que a justifica.

As três features mais importantes são os **lags de carga**: média móvel
24h, lag 168h (semana passada) e lag 24h (ontem mesma hora). O sistema
elétrico tem memória forte — o melhor preditor da carga agora é a carga
recente. Isso é esperado e confirma que o Seasonal Naive, apesar de
simples, capturava a informação mais importante.

O **dia da semana e a hora** aparecem logo em seguida — o perfil
intradiário e a distinção dia útil/fim de semana são determinantes, como
as EDAs anteriores mostraram.

A **temperatura** e seus lags (3h e 24h) ocupam posições intermediárias
mas relevantes — confirmando que a análise de sensibilidade térmica
identificou as features certas. O lag de 3h superando a temperatura
instantânea valida empiricamente o efeito de inércia térmica descoberto
na EDA de temperatura.

O **feriado** aparece acima das variáveis cíclicas — o modelo aprendeu
que feriados quebram o padrão de forma não capturável por hora ou dia
da semana isoladamente.

O **é_fds** ter a menor importância é contraintuitivo, mas faz sentido:
o dia da semana já captura sábado e domingo explicitamente. A variável
binária de fim de semana é redundante quando dia_semana está presente.

Com os três modelos treinados e as features validadas, o último passo é
comparar a performance de forma consolidada — visualizando o erro de cada
modelo ao longo do tempo e região a região.

## 6. Comparação dos Modelos

*Heatmap de MAPE por modelo e região. Evolução do erro ao longo do tempo.*

In [ ]:
# ── Tabela comparativa de MAPE ────────────────────────────────────────
import plotly.figure_factory as ff

regioes_ord = ['NORTE', 'NORDESTE', 'SUDESTE', 'SUL']

mape_matrix = {
    'Seasonal Naive': {r: results_naive[r]['MAPE (%)'] for r in regioes_ord},
    'Regressão Linear': {r: results_lr[r]['MAPE (%)'] for r in regioes_ord},
    'XGBoost':  {r: results_xgb[r]['MAPE (%)'] for r in regioes_ord},
}

df_comp = pd.DataFrame(mape_matrix).T
df_comp['Média'] = df_comp.mean(axis=1)

# ── Heatmap de MAPE ───────────────────────────────────────────────────
fig = go.Figure(go.Heatmap(
    z=df_comp.values,
    x=list(df_comp.columns),
    y=list(df_comp.index),
    colorscale='RdYlGn_r',
    text=[[f'{v:.2f}%' for v in row] for row in df_comp.values],
    texttemplate='%{text}',
    textfont=dict(size=13, color='white'),
    hovertemplate='Modelo: %{y}<br>Região: %{x}<br>MAPE: %{z:.2f}%<extra></extra>',
    showscale=True,
    colorbar=dict(title='MAPE (%)', thickness=15)
))

fig.update_layout(
    title='<b>MAPE por Modelo e Região</b><br><sup>Verde = melhor performance</sup>',
    template='plotly_dark',
    paper_bgcolor='#0d1117',
    plot_bgcolor='#0d1117',
    height=300,
    margin=dict(l=150, r=60, t=80, b=60),
    xaxis=dict(title='', tickfont=dict(size=12)),
    yaxis=dict(title='', tickfont=dict(size=12)))

fig.show()
print(df_comp.round(2))


### Observações

O heatmap consolida em uma única visualização o que os três modelos
entregaram.

O XGBoost domina em todas as regiões sem exceção — linha inteiramente
verde, MAPE entre 2.04% e 3.88%. O Sudeste com 2.04% é o resultado mais
expressivo, considerando a complexidade e o volume de carga da região.

O caso mais revelador é o Norte: o Seasonal Naive (3.46%) superou a
Regressão Linear (5.24%), mas o XGBoost (2.31%) superou ambos com
folga. É a prova de que a relação não-linear existe e o modelo certo
consegue capturá-la — enquanto o modelo errado piora o resultado.

O Sul em vermelho no Naive (8.92%) e verde escuro no XGBoost (3.88%)
é o maior salto de performance da tabela — 5 pontos percentuais de
ganho numa região que a análise de temperatura mostrou ser a mais
complexa e sensível a interações entre variáveis.

A linha da Regressão Linear fica no meio — melhor que o Naive em 3 de
4 regiões, mas muito atrás do XGBoost em todas. É o papel esperado de
um baseline de modelo: melhor que não ter modelo, mas longe do potencial
real dos dados.

## Conclusão

Este notebook fecha uma série de quatro análises sobre o consumo de energia
elétrica no Brasil — e responde a pergunta que as três anteriores
prepararam: **conseguimos prever a carga horária do SIN?**

A resposta é sim, com qualidade.

O XGBoost atingiu MAPE médio de 2.77% nos quatro subsistemas, com destaque
para o Sudeste (2.04%) — resultado expressivo para previsão de carga
horária com dado real, sem ajuste fino de hiperparâmetros.

A progressão dos modelos conta uma história clara. O Seasonal Naive (5.7%)
mostrou que o padrão semanal explica muito — mas falha em feriados e
clima atípico. A Regressão Linear (4.65%) melhorou ao incluir temperatura
e calendário, mas não conseguiu capturar interações não-lineares — o Norte
piorou em relação ao Naive. O XGBoost (2.77%) resolveu ambos os problemas:
capturou as não-linearidades que a regressão não via e aprendeu o efeito
de feriados que o Naive ignorava.

O feature importance confirmou os insights analíticos das EDAs anteriores:
os lags de carga dominam (o sistema tem memória forte), temperatura e seus
lags aparecem na posição esperada com o lag de 3h superando a temperatura
instantânea — validando empiricamente o efeito de inércia térmica descoberto
na análise de sensibilidade —, e o feriado se mostrou mais informativo do
que as variáveis cíclicas.

A variável é_fds, redundante com dia_semana, teve importância mínima —
um lembrete de que feature engineering sem análise pode introduzir ruído
em vez de sinal.

